In [ ]:
import numpy as np
import pandas as pd
import os

# 1. 데이터 생성 핵심 함수 (기준 파일 로직)
def generate_raw_items(n_students, n_items, target_mean, target_std, min_val, max_val, is_sum=False):
    """
    학생별 목표 점수를 생성한 뒤 문항별로 분배하고, 합계 오차를 정밀하게 조정하는 함수
    """
    # 학생별 목표 점수 생성
    scores = np.random.randn(n_students)
    scores = (scores - np.mean(scores)) / np.std(scores) * target_std + target_mean
    
    data = np.zeros((n_students, n_items))
    for i in range(n_students):
        # is_sum=True인 경우 scores[i]를 총점으로, False인 경우 평균으로 취급
        target_sum = scores[i] if is_sum else scores[i] * n_items
        
        # 임의 분배 후 스케일링
        row = np.random.rand(n_items) + 0.5 
        row = (row / np.sum(row)) * target_sum
        row = np.clip(np.round(row), min_val, max_val)
        
        # 정수 반올림으로 인한 합계 오차 조정 루프
        diff = int(round(target_sum)) - int(np.sum(row))
        attempts = 0
        while diff != 0 and attempts < 1000:
            idx = np.random.randint(n_items)
            if diff > 0 and row[idx] < max_val:
                row[idx] += 1
                diff -= 1
            elif diff < 0 and row[idx] > min_val:
                row[idx] -= 1
                diff += 1
            attempts += 1
        data[i] = row
    return data.astype(int)

# 2. 파라미터 설정 및 데이터 생성
n_per_group = 18
factors = ['Fluency', 'Accuracy', 'Interaction', 'Content']

# 말하기 성취도 통계치 (is_sum=True 사용)
spk_pre_exp = generate_raw_items(n_per_group, 4, 13.45, 1.82, 1, 5, is_sum=True)
spk_post_exp = generate_raw_items(n_per_group, 4, 16.92, 1.65, is_sum=True)
spk_pre_ctrl = generate_raw_items(n_per_group, 4, 13.50, 1.80, 1, 5, is_sum=True)
spk_post_ctrl = generate_raw_items(n_per_group, 4, 13.55, 1.82, 1, 5, is_sum=True)

# 3. 데이터프레임 구축 함수
def build_speaking_df(group_name, spk_pre, spk_post, start_id):
    df = pd.DataFrame()
    df['Student_ID'] = [f'{group_name[0].upper()}{str(i).zfill(2)}' for i in range(start_id, start_id + n_per_group)]
    df['Group'] = group_name
    
    for i, factor in enumerate(factors):
        df[f'Spk_Pre_{factor}'] = spk_pre[:, i]
        df[f'Spk_Post_{factor}'] = spk_post[:, i]
    
    # 총점 계산
    df['Spk_Pre_Total'] = df[[f'Spk_Pre_{f}' for f in factors]].sum(axis=1)
    df['Spk_Post_Total'] = df[[f'Spk_Post_{f}' for f in factors]].sum(axis=1)
    return df

# 데이터 통합 및 저장
df_final = pd.concat([
    build_speaking_df('Experimental', spk_pre_exp, spk_post_exp, 1),
    build_speaking_df('Control', spk_pre_ctrl, spk_post_ctrl, 1)
], ignore_index=True)

df_final.to_csv('simulated_speaking_data.csv', index=False)

# 4. 검증 (기술통계량 확인)
print("=== 생성 데이터 검증 (말하기 총점 평균) ===")
print(df_final.groupby('Group')[['Spk_Pre_Total', 'Spk_Post_Total']].agg(['mean', 'std']).round(2))

✅ 말하기 데이터 생성 완료! 파일 경로: ./speaking/simulated_speaking_data.csv
📊 데이터 요약 (실험집단 사후 평균): 16.94
